In [1]:
stop
!pip install requests-html

NameError: name 'stop' is not defined

In [ ]:
!pip install "lxml[html_clean]"

In [ ]:
!pip install nest_asyncio

In [ ]:
!pip install selenium pandas

In [ ]:
!pip install webdriver_manager

In [ ]:
import requests

url = "https://repeatsdb.org/api/production/annotations"
params = {
    "skip": 0,
    "limit": 25,
    "updated.by": "user,predictor,mapping"
}
headers = {
    "Cache-Control": "no-cache"  # Force fresh data instead of cached
}

response = requests.get(url, params=params, headers=headers)
print("Status Code:", response.status_code)

# If the response is JSON, you can load it as follows:
try:
    data = response.json()
    print("Data received:", data)
except Exception as e:
    print("Error parsing JSON:", e)

In [ ]:
import asyncio
import pandas as pd
from requests_html import AsyncHTMLSession
from tqdm import tqdm
import time

async def scrape_repeatsdb_api(
    updated_by="user,predictor,mapping",
    output_file="repeatsdb_annotations.csv",
    limit=100,
    max_pages=None,
    max_retries=3
):
    """
    Scrape annotations from the RepeatsDB API using no-cache headers and update a single tqdm progress bar.
    """
    asession = AsyncHTMLSession()
    base_url = "https://repeatsdb.org/api/production/annotations"
    headers = {
        "Cache-Control": "no-cache",
        "User-Agent": "Mozilla/5.0 (compatible; YourScraper/1.0)"
    }
    
    page_number = 1
    wrote_headers = False
    
    # If max_pages is provided, set total; otherwise, leave total as None for an indeterminate progress bar.
    pbar = tqdm(total=max_pages if max_pages else None, desc="Scraping API pages", dynamic_ncols=True)
    
    while True:
        skip = (page_number - 1) * limit
        params = {
            "skip": skip,
            "limit": str(limit),
            "updated.by": updated_by
        }
        
        # Implement retry logic for network requests.
        for attempt in range(1, max_retries + 1):
            try:
                response = await asession.get(base_url, params=params, headers=headers)
                break  # If the request is successful, exit the retry loop.
            except Exception as e:
                if attempt == max_retries:
                    pbar.write(f"Max retries reached on page {page_number}. Exiting.")
                    pbar.close()
                    return
                await asyncio.sleep(2 ** attempt)  # Exponential backoff.
        
        if response.status_code != 200:
            pbar.write(f"Error: Received status code {response.status_code} on page {page_number}.")
            break
        
        try:
            data = response.json()
        except Exception as e:
            pbar.write(f"Error parsing JSON on page {page_number}: {e}")
            break
        
        # End loop if no data is returned.
        if not data:
            pbar.write("No data returned. Ending pagination.")
            break
        
        # Convert JSON list to DataFrame.
        df = pd.DataFrame(data)
        if df.empty:
            pbar.write("Empty dataframe returned, ending pagination.")
            break
        
        # Write to CSV (write headers on first page, then append).
        if not wrote_headers:
            df.to_csv(output_file, index=False, mode="w", header=True)
            wrote_headers = True
        else:
            df.to_csv(output_file, index=False, mode="a", header=False)
        
        # Update the single tqdm progress bar.
        pbar.set_postfix({"page": page_number, "rows": len(df)})
        pbar.update(1)
        
        page_number += 1
        if max_pages and page_number > max_pages:
            pbar.write(f"Reached max_pages limit of {max_pages}. Stopping.")
            break
        
        # Delay before the next request.
        await asyncio.sleep(1)
    
    pbar.close()

# Run the scraper in an async environment.
await scrape_repeatsdb_api(max_pages=454)

In [ ]:
#!/usr/bin/env python3
import csv
import ast
import sys

# Increase the CSV field size limit to handle larger fields
csv.field_size_limit(sys.maxsize)

# Input and output file paths
input_file = "/mnt/home/pkwon2/STRPsearch/repeatsdb_annotations.csv"
output_file = "/mnt/home/pkwon2/STRPsearch/repeatsdb_annotations_clean.csv"

# Define the header for the clean CSV (added region_classes)
fieldnames = ["start", "end", "class", "region_classes", "structure", "id", "source", "reviewed"]

with open(input_file, "r", newline="", encoding="utf-8") as fin, \
     open(output_file, "w", newline="", encoding="utf-8") as fout:

    reader = csv.DictReader(fin)
    writer = csv.DictWriter(fout, fieldnames=fieldnames)
    writer.writeheader()
    
    for row in reader:
        # The original CSV has a column 'items' that holds a dict as a string
        try:
            record = ast.literal_eval(row["items"])
        except Exception as e:
            print(f"Skipping row due to parse error: {e}")
            continue

        content = record.get("content", {})
        loci = content.get("loci", [])
        if not loci:
            print("No loci found for record, skipping.")
            continue

        # Calculate the overall smallest start and largest end among all loci entries.
        try:
            min_start = min(int(locus.get("start")) for locus in loci if locus.get("start") is not None)
            max_end = max(int(locus.get("end")) for locus in loci if locus.get("end") is not None)
        except Exception as e:
            print(f"Error computing boundaries: {e}")
            continue

        # Collect all the classes from loci with type 'region'
        region_classes_list = [
            locus.get("class", "") 
            for locus in loci 
            if locus.get("type") == "region" and locus.get("class")
        ]
        region_classes = ", ".join(region_classes_list)
        
        # Also get the first region's class (if any) to use in the "class" field.
        region_class = ""
        for locus in loci:
            if locus.get("type") == "region" and locus.get("class"):
                region_class = locus.get("class")
                break

        # Extract chain data
        chain = content.get("chain", {})

        # Assemble the new row with recalculated start/end and added region_classes
        new_row = {
            "start": str(min_start),
            "end": str(max_end),
            "class": region_class,
            "region_classes": region_classes,
            "structure": chain.get("structure"),
            "id": chain.get("id"),
            "source": chain.get("source"),
            "reviewed": content.get("reviewed")
        }
        writer.writerow(new_row)

In [ ]:
#!/usr/bin/env python3
import csv
import random

# Input and output file paths
input_file = "/mnt/home/pkwon2/STRPsearch/repeatsdb_annotations_clean.csv"
output_file = "/mnt/home/pkwon2/STRPsearch/duplicates_dropped.csv"

# Dictionary to hold rows by the "structure" column
rows_by_structure = {}

with open(input_file, "r", newline="", encoding="utf-8") as fin:
    reader = csv.DictReader(fin)
    header = reader.fieldnames  # keep the header from the original CSV
    for row in reader:
        structure = row.get("structure")
        if structure in rows_by_structure:
            rows_by_structure[structure].append(row)
        else:
            rows_by_structure[structure] = [row]

# For each group, choose a random row
selected_rows = []
for structure, rows in rows_by_structure.items():
    selected_row = random.choice(rows)
    selected_rows.append(selected_row)

# Write the selected rows to a new CSV file
with open(output_file, "w", newline="", encoding="utf-8") as fout:
    writer = csv.DictWriter(fout, fieldnames=header)
    writer.writeheader()
    writer.writerows(selected_rows)

print(f"Duplicates dropped CSV created: {output_file}")

In [ ]:
#!/usr/bin/env python3
import pandas as pd

def main():
    # Define file paths
    file_path1 = '/home/pkwon2/STRPsearch/duplicates_dropped.csv'
    file_path2 = '/home/pkwon2/STRPsearch/duplicates_dropped_backup.csv'
    
    # Read the CSV files
    df_primary = pd.read_csv(file_path1)
    df_backup = pd.read_csv(file_path2)
    
    # Merge the dataframes on the "structure" column
    # Select the desired columns from each dataframe:
    #   - From duplicates_dropped.csv: structure, start, end, region_classes
    #   - From duplicates_dropped_backup.csv: structure, id
    df_merged = pd.merge(
        df_primary[['structure', 'start', 'end', 'region_classes', 'source', 'reviewed']],
        df_backup[['structure', 'id']],
        on='structure',
        how='inner'  # Change to 'left' or 'outer' if needed
    )
    
    # Write the merged dataframe to a new CSV file
    output_file = '/home/pkwon2/STRPsearch/merged.csv'
    df_merged.to_csv(output_file, index=False)
    print(f"Merged CSV saved to {output_file}")

if __name__ == '__main__':
    main()

In [ ]:
#!/usr/bin/env python3
import os
import pandas as pd
import requests
import time

def get_rcsb_info(pdb_id):
    """
    Query the RCSB PDB API for the provided pdb_id and extract:
      - Resolution (from rcsb_entry_info.resolution_combined)
      - R-Value Free, R-Value Work, R-Value Observed (from rcsb_refinement or pdbx_vrpt)
    """
    url = f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id.upper()}"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            # Extract resolution from rcsb_entry_info
            resolution = data.get("rcsb_entry_info", {}).get("resolution_combined", [None])[0]
            
            # Initialize R-value fields
            r_work = None
            r_free = None
            r_obs = None

            # Attempt to extract refinement details from rcsb_refinement
            refinement_data = data.get("rcsb_refinement")
            if refinement_data:
                if isinstance(refinement_data, list) and len(refinement_data) > 0:
                    block = refinement_data[0]
                elif isinstance(refinement_data, dict):
                    block = refinement_data
                else:
                    block = {}
                r_work = block.get("r_work")
                r_free = block.get("r_free")
                # Try both key names for observed value
                r_obs = block.get("r_value_obs", block.get("r_value_observed"))
            
            # Fallback: Try to extract from pdbx_vrpt if no r_work found from the refinement block
            if r_work is None:
                vrpt_data = data.get("pdbx_vrpt")
                if vrpt_data:
                    if isinstance(vrpt_data, list) and len(vrpt_data) > 0:
                        block = vrpt_data[0]
                    elif isinstance(vrpt_data, dict):
                        block = vrpt_data
                    else:
                        block = {}
                    r_work = block.get("r_work")
                    r_free = block.get("r_free")
                    r_obs = block.get("r_value_obs", block.get("r_value_observed"))
            
            return resolution, r_free, r_work, r_obs
        else:
            print(f"Failed to fetch data for {pdb_id}. Status code: {response.status_code}")
            return None, None, None, None
    except Exception as e:
        print(f"Error fetching data for {pdb_id}: {e}")
        return None, None, None, None

def main():
    # File paths
    input_file = '/home/pkwon2/STRPsearch/merged.csv'
    checkpoint_file = '/home/pkwon2/STRPsearch/merged_with_rcsb_info.csv'
    
    # Load from checkpoint if it exists; otherwise load the original CSV and add new columns
    if os.path.exists(checkpoint_file):
        print(f"Loading checkpoint from {checkpoint_file}")
        df = pd.read_csv(checkpoint_file)
    else:
        print(f"Loading original CSV from {input_file}")
        df = pd.read_csv(input_file)
        # Add new columns if they don't already exist
        for col in ['resolution', 'r_value_free', 'r_value_work', 'r_value_observed']:
            if col not in df.columns:
                df[col] = None

    # Process each row, only for those that haven't been updated
    for index, row in df.iterrows():
        if "RCSB/PDB" in str(row.get("source", "")):
            structure_id = row.get("structure")
            # Process if structure_id exists and hasn't been processed yet
            if pd.notnull(structure_id) and pd.isnull(row.get("resolution")):
                print(f"Processing structure {structure_id} at index {index}...")
                resolution, r_free, r_work, r_obs = get_rcsb_info(structure_id)
                # Print out the retrieved values in the desired format
                print(f"Resolution: {resolution} | R-Value Free: {r_free} | R-Value Work: {r_work} | R-Value Observed: {r_obs}")
                df.at[index, "resolution"] = resolution
                df.at[index, "r_value_free"] = r_free
                df.at[index, "r_value_work"] = r_work
                df.at[index, "r_value_observed"] = r_obs
                
                # Write the updated DataFrame to disk as a checkpoint
                df.to_csv(checkpoint_file, index=False)
                # Sleep briefly to avoid overloading the server
                time.sleep(1)
                
    print(f"Processing complete. Checkpoint file saved to {checkpoint_file}")

if __name__ == '__main__':
    main()

In [ ]:
#!/bin/bash
#SBATCH -p cpu
#SBATCH -J mmseqs_iiab
#SBATCH --time=02:00:00
#SBATCH -c 8
#SBATCH --mem=12g
#SBATCH -o digs.log
#SBATCH -e digs.log

/software/mmseqs2/bin/mmseqs easy-linclust PATH/TO/INPUT/FILE.fasta PATH/TO/OUTPUT/PREFIX /tmp --min-seq-id 0.98 -c 0.8 --cov-mode 0 --threads 8

In [ ]:
#!/usr/bin/env python3
import csv

# Input and output file paths
input_file = "/mnt/home/pkwon2/STRPsearch/duplicates_dropped.csv"
tasks_file = "/mnt/home/pkwon2/STRPsearch/tasks.txt"

with open(input_file, "r", newline="", encoding="utf-8") as fin, \
     open(tasks_file, "w", newline="", encoding="utf-8") as fout:
    
    reader = csv.DictReader(fin)
    
    for row in reader:
        structure = row.get("structure", "").strip()
        chain_id = row.get("id", "").strip()
        source = row.get("source", "").strip()
        
        if "RCSB/PDB" in source:
            # For RCSB/PDB, include the --chain parameter.
            command = f"python3 ./bin/strpsearch.py download-pdb {structure} /net/scratch/pkwon2/repeatsdb/{structure} --chain {chain_id}"
        elif "AlphaFoldDB" in source:
            # For AlphaFoldDB, no chain parameter is needed.
            command = f"python3 ./bin/strpsearch.py download-model {structure} /net/scratch/pkwon2/repeatsdb/{structure}"
        else:
            # If the source doesn't match, skip this row.
            continue
        
        fout.write(command + "\n")

print(f"Tasks file created at: {tasks_file}")

In [ ]:
#!/bin/bash
# Change to the STRPsearch directory
cd /home/pkwon2/STRPsearch

# Ensure tasks.txt exists
if [ ! -f tasks.txt ]; then
    echo "tasks.txt not found in $(pwd)"
    exit 1
fi

# Split tasks.txt into chunks of 10 lines each (adjust the number if needed)
mkdir -p splits
split -l 10 tasks.txt splits/x

# Count the number of split files (chunks)
NUM_CHUNKS=$(ls -1 splits/x* | wc -l)
echo "Number of chunks: $NUM_CHUNKS"

# Create a job array script (run_array.sh)
cat << 'EOF' > run_array.sh
#!/bin/bash
#SBATCH --job-name=strpsearch_parallel
#SBATCH --output=slurm-%A_%a.out
#SBATCH --error=slurm-%A_%a.err
#SBATCH --cpus-per-task=4
#SBATCH --mem=4g
#SBATCH --time=1:00:00
#SBATCH --array=1-NUM_CHUNKS

# Change to the STRPsearch directory
cd /home/pkwon2/STRPsearch

# Initialize conda in the shell and activate the environment
eval "$(conda shell.bash hook)"
conda activate strpsearch_env

# Get list of chunk files sorted (SLURM_ARRAY_TASK_ID is 1-indexed)
chunks=(splits/x*)
chunk=${chunks[$SLURM_ARRAY_TASK_ID-1]}
echo "Running chunk file: $chunk"
bash "$chunk"
EOF

# Replace NUM_CHUNKS placeholder with actual number
sed -i "s/NUM_CHUNKS/$NUM_CHUNKS/g" run_array.sh

# Submit the job array
sbatch run_array.sh

# Optionally, check job status with your scheduler command (e.g., sjobs)
sjobs

In [ ]:
#!/usr/bin/env python3

import os
import csv
import shutil
import glob
from tqdm import tqdm

# Define paths.
csv_path = "/home/pkwon2/STRPsearch/duplicates_dropped.csv"
dest_dir = "/projects/ml/pkwon2/repeats_rcsb/"

# Create the destination directory if it doesn't exist.
os.makedirs(dest_dir, exist_ok=True)

# Open and process the CSV file.
with open(csv_path, newline='') as csvfile:
    reader = csv.DictReader(csvfile)
    rows = list(reader)  # Convert reader to a list for total count
    for row in tqdm(rows, desc="Processing rows"):
        # Check if the 'source' column contains "RCSB".
        if "RCSB" in row.get('source', ''):
            structure = row.get('structure', '').strip()
            if not structure:
                continue  # Skip if structure is empty.

            # Construct the path to the expected results directory.
            results_dir = os.path.join("/net/scratch/pkwon2/repeatsdb", structure, f"{structure}_results")
            if os.path.isdir(results_dir):
                # Construct the source directory for query PDB files.
                query_dir = os.path.join("/net/scratch/pkwon2/repeatsdb", structure, "query_structures")
                # Use glob to find all files ending with .pdb in the query directory.
                pdb_files = glob.glob(os.path.join(query_dir, "*.pdb"))
                if pdb_files:
                    for pdb_file in pdb_files:
                        try:
                            shutil.copy(pdb_file, dest_dir)
                        except Exception as e:
                            print(f"Error copying {pdb_file}: {e}")
                else:
                    print(f"No pdb files found in {query_dir}")

In [ ]:
#!/usr/bin/env python3
import os
import csv
import glob
import shutil
from tqdm import tqdm

# Paths
csv_path = '/home/pkwon2/STRPsearch/duplicates_dropped.csv'
dest_dir = '/net/scratch/pkwon2/repeats_alphafold/'

# Create destination directory if it doesn't exist
if not os.path.exists(dest_dir):
    os.makedirs(dest_dir)

# Read CSV rows
with open(csv_path, newline='') as csvfile:
    rows = list(csv.DictReader(csvfile))

# First pass: calculate the total number of PDB files to copy
total_files = 0
for row in rows:
    if 'AlphaFold' in row.get('source', ''):
        structure = row.get('structure')
        if structure:
            pdb_pattern = f'/net/scratch/pkwon2/repeatsdb/{structure}/query_structures/*pdb'
            pdb_files = glob.glob(pdb_pattern)
            total_files += len(pdb_files)

# Create a single tqdm progress bar for file copying
with tqdm(total=total_files, desc="Copying PDB files") as pbar:
    # Second pass: copy each PDB file and update the progress bar
    for row in rows:
        if 'AlphaFold' in row.get('source', ''):
            structure = row.get('structure')
            if not structure:
                continue  # Skip rows without structure information
            pdb_pattern = f'/net/scratch/pkwon2/repeatsdb/{structure}/query_structures/*pdb'
            pdb_files = glob.glob(pdb_pattern)
            if not pdb_files:
                print(f"No PDB files found for structure: {structure}")
            for pdb_file in pdb_files:
                try:
                    shutil.copy(pdb_file, dest_dir)
                except Exception as e:
                    print(f"Failed to copy {pdb_file}: {e}")
                finally:
                    pbar.update(1)

In [ ]:
import os
import csv
import glob

# Define directories and CSV file path
rcsb_dir = '/net/scratch/pkwon2/repeats_rcsb'
# alphafold_dir = '/net/scratch/pkwon2/repeats_alphafold'
csv_file = '/home/pkwon2/STRPsearch/merged.csv'

# Checkpoint file to record processed pdbs
checkpoint_file = 'processed_checkpoints.txt'

def load_checkpoint(checkpoint_path):
    """Load the set of processed pdb file paths from the checkpoint file."""
    processed = set()
    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, 'r') as cf:
            for line in cf:
                processed.add(line.strip())
    return processed

def update_checkpoint(checkpoint_path, pdb_path):
    """Append the processed pdb file path to the checkpoint file."""
    with open(checkpoint_path, 'a') as cf:
        cf.write(pdb_path + "\n")

def process_pdb(pdb_file, start, end, search_dir):
    """Process a pdb file:
       - For ATOM records, keep only atoms whose residue number is within [start, end].
       - For HETATM records in RCSB pdb files, remove all heteroatoms.
       - For HETATM records in other pdb files, keep them unaltered.
       - Other lines (e.g. TER, END) remain unaltered.
    """
    try:
        with open(pdb_file, 'r') as f:
            lines = f.readlines()
    except Exception as e:
        print(f"Failed to open {pdb_file}: {e}")
        return False

    new_lines = []
    for line in lines:
        if line.startswith("ATOM"):
            # Residue number is typically in columns 23-26 (0-indexed 22:26)
            try:
                resnum_str = line[22:26].strip()
                resnum = int(resnum_str)
            except ValueError:
                # In case of any conversion issues, keep the line unmodified.
                new_lines.append(line)
                continue

            # Keep only atoms in the desired residue range.
            if start <= resnum <= end:
                new_lines.append(line)
            else:
                # Skip atoms outside of the range.
                continue
        elif line.startswith("HETATM"):
            if search_dir == rcsb_dir:
                # Remove all HETATM lines for RCSB pdb files.
                continue
            else:
                # For Alphafold pdbs or other sources, keep heteroatom records.
                new_lines.append(line)
        else:
            # For other lines (TER, END, etc.) simply keep them.
            new_lines.append(line)

    try:
        # Write the processed content back to the same file.
        with open(pdb_file, 'w') as f:
            f.writelines(new_lines)
        print(f"Processed: {pdb_file}")
        return True
    except Exception as e:
        print(f"Failed to write processed file {pdb_file}: {e}")
        return False

def main():
    processed_files = load_checkpoint(checkpoint_file)

    # Read the CSV
    with open(csv_file, newline='') as csvf:
        reader = csv.DictReader(csvf)
        for row in reader:
            try:
                start = int(row["start"])
                end = int(row["end"])
            except ValueError:
                print(f"Skipping row with invalid start/end values: {row}")
                continue

            source = row["source"].strip()
            structure = row["structure"].strip()

            # Determine which directory to search based on the source.
            if "RCSB" in source:
                search_dir = rcsb_dir
            elif "AlphaFold" in source:
                search_dir = alphafold_dir
            else:
                print(f"Unknown source '{source}' for structure '{structure}'. Skipping row.")
                continue

            # Glob for pdb files that match the structure pattern.
            pdb_pattern = os.path.join(search_dir, f"*{structure}*.pdb")
            pdb_files = glob.glob(pdb_pattern)

            if not pdb_files:
                print(f"No PDB files found for pattern: {pdb_pattern}")
                continue

            for pdb_file in pdb_files:
                # If already processed, skip this file.
                if pdb_file in processed_files:
                    print(f"Skipping already processed file: {pdb_file}")
                    continue

                # Process the pdb file
                if process_pdb(pdb_file, start, end, search_dir):
                    # Update the checkpoint upon successful processing.
                    update_checkpoint(checkpoint_file, pdb_file)
                    processed_files.add(pdb_file)

if __name__ == '__main__':
    main()

In [ ]:
#!/usr/bin/env python3
import os
import glob

# Directories containing pdb files
rcsb_dir = '/net/scratch/pkwon2/repeats_rcsb'
# alphafold_dir = '/net/scratch/pkwon2/repeats_alphafold'
# List of directories to process
dirs_to_process = [rcsb_dir, alphafold_dir]

# Checkpoint filename (stored in the same folder as the script)
CHECKPOINT_FILE = "processed_checkpoint.txt"

# Load checkpointed files into a set for quick lookup.
def load_checkpoint(checkpoint_path):
    processed = set()
    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, "r") as f:
            for line in f:
                processed.add(line.strip())
    return processed

# Append a file record to the checkpoint file.
def update_checkpoint(checkpoint_path, file_record):
    with open(checkpoint_path, "a") as f:
        f.write(f"{file_record}\n")

# Process a single pdb file.
def process_pdb_file(filepath):
    try:
        with open(filepath, "r") as f:
            lines = f.readlines()
    except Exception as e:
        print(f"Error reading file {filepath}: {e}")
        return False  # processing failed

    processed_lines = []
    current_resnum = 0
    last_residue_key = None  # used to track boundary between residues

    # Iterate through each line and process ATOM/HETATM records.
    for line in lines:
        if line.startswith("ATOM") or line.startswith("HETATM"):
            # Extract residue identifier components:
            # columns (using 0-indexed slicing):
            # resname: columns 17-20, resSeq: columns 22-26, insertion code: column 26.
            resname = line[17:20].strip()
            resseq = line[22:26].strip()
            icode  = line[26].strip()  # insertion code, replace with blank later
            residue_key = (resname, resseq, icode)
            # Check if a new residue is encountered.
            if residue_key != last_residue_key:
                current_resnum += 1
                last_residue_key = residue_key

            # Build the new line:
            # The chain id is in column 22 (index 21) and will be replaced with "A".
            # The residue sequence number spans columns 23-26.
            # We reset the insertion code (column 27, index 26) to a blank.
            new_resseq = f"{current_resnum:>4}"
            # Construct the new record: keep columns up to chain id, replace chain id with "A",
            # then use the new residue number, then place a blank for insertion code and keep the rest.
            new_line = line[:21] + "A" + new_resseq + " " + line[27:]
            processed_lines.append(new_line)
        else:
            # For non ATOM/HETATM lines, copy as is.
            processed_lines.append(line)

    # Decide whether to delete the file based on residue count.
    if current_resnum > 9999:
        print(f"Deleting {filepath} (residues: {current_resnum})")
        try:
            os.remove(filepath)
        except Exception as e:
            print(f"Error deleting file {filepath}: {e}")
            return False  # mark file as not processed if deletion fails
    else:
        # Overwrite the original file with processed data.
        try:
            # It is safer to write to a temporary file then rename.
            tmp_filepath = filepath + ".tmp"
            with open(tmp_filepath, "w") as f:
                f.writelines(processed_lines)
            os.replace(tmp_filepath, filepath)
            print(f"Processed {filepath} (residues: {current_resnum})")
        except Exception as e:
            print(f"Error writing processed file {filepath}: {e}")
            return False
    return True

def main():
    processed_files = load_checkpoint(CHECKPOINT_FILE)
    
    # For each directory
    for directory in dirs_to_process:
        # Use glob to list pdb files (assuming extension .pdb)
        pdb_files = glob.glob(os.path.join(directory, "*.pdb"))
        for pdb_file in pdb_files:
            abs_path = os.path.abspath(pdb_file)
            if abs_path in processed_files:
                print(f"Skipping already processed file: {abs_path}")
                continue
            success = process_pdb_file(abs_path)
            # If processing (or deletion) succeeded, add to checkpoint so that we don't reprocess.
            if success:
                update_checkpoint(CHECKPOINT_FILE, abs_path)

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
import os
import glob

# Directories to process
rcsb_dir = '/net/scratch/pkwon2/repeats_rcsb'
alphafold_dir = '/net/scratch/pkwon2/repeats_alphafold'
dirs_to_process = [rcsb_dir, alphafold_dir]

def filter_atoms_in_file(filepath):
    """
    Reads a pdb file, filters out all lines not starting with 'ATOM',
    and writes back only the matching lines using a temporary file for safety.
    """
    try:
        with open(filepath, "r") as f:
            lines = f.readlines()
    except Exception as e:
        print(f"Error reading file {filepath}: {e}")
        return False

    # Filter lines to keep only those starting with 'ATOM'
    atom_lines = [line for line in lines if line.startswith("ATOM")]

    try:
        tmp_filepath = filepath + ".tmp"
        with open(tmp_filepath, "w") as f:
            f.writelines(atom_lines)
        os.replace(tmp_filepath, filepath)
        print(f"Processed file: {filepath}. Kept {len(atom_lines)} ATOM lines.")
    except Exception as e:
        print(f"Error writing file {filepath}: {e}")
        return False

    return True

def main():
    # Process each directory
    for directory in dirs_to_process:
        pdb_files = glob.glob(os.path.join(directory, "*.pdb"))
        for pdb_file in pdb_files:
            abs_path = os.path.abspath(pdb_file)
            filter_atoms_in_file(abs_path)

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
import os
import glob

# Define the target directory for pdb files.
target_dir = '/net/scratch/pkwon2/repeats_rcsb'

def delete_empty_atom_pdbs(directory):
    # Find all PDB files in the directory.
    pdb_files = glob.glob(os.path.join(directory, "*.pdb"))
    
    for pdb_file in pdb_files:
        try:
            with open(pdb_file, "r") as f:
                # Count lines that start with "ATOM"
                atom_count = sum(1 for line in f if line.startswith("ATOM"))
            
            # If the file has 0 ATOM lines, delete it.
            if atom_count == 0:
                print(f"Deleting {pdb_file} (0 ATOM lines)")
                os.remove(pdb_file)
            else:
                print(f"Keeping {pdb_file} (ATOM lines: {atom_count})")
        except Exception as e:
            print(f"Error processing file {pdb_file}: {e}")

def main():
    delete_empty_atom_pdbs(target_dir)

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
import os
import glob
import csv

# Path to the CSV file
csv_file = '/home/pkwon2/STRPsearch/merged_with_rcsb_info.csv'
# Directory containing the structure files
target_dir = '/net/scratch/pkwon2/repeats_rcsb'

# Open and read the CSV file
with open(csv_file, newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        # Get structure identifier and the resolution string from the CSV
        structure = row.get('structure')
        resolution_str = row.get('resolution', '')

        # Try to convert the resolution to float. If conversion fails, mark it as invalid.
        try:
            resolution = float(resolution_str)
        except (ValueError, TypeError):
            print(f"Invalid resolution for structure {structure}: {resolution_str}")
            resolution = None

        # Delete files for structures with invalid resolution or resolution >= 2.5
        if resolution is None or resolution >= 2.5:
            # Create a pattern to match files containing the structure identifier and ending with 'pdb'
            pattern = os.path.join(target_dir, f"*{structure}*pdb")
            matching_files = glob.glob(pattern)
            for file_path in matching_files:
                try:
                    os.remove(file_path)
                    print(f"Removed file: {file_path}")
                except Exception as e:
                    print(f"Error removing {file_path}: {e}")

In [ ]:
import os
import csv
import glob

# Define directories and CSV file path (only the RCSB directory is used)
rcsb_dir = '/net/scratch/pkwon2/repeats_rcsb'
csv_file = '/home/pkwon2/STRPsearch/merged.csv'

# Checkpoint file to record processed pdbs
checkpoint_file = 'processed_checkpoints.txt'

def load_checkpoint(checkpoint_path):
    """Load the set of processed pdb file paths from the checkpoint file."""
    processed = set()
    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, 'r') as cf:
            for line in cf:
                processed.add(line.strip())
    return processed

def update_checkpoint(checkpoint_path, pdb_path):
    """Append the processed pdb file path to the checkpoint file."""
    with open(checkpoint_path, 'a') as cf:
        cf.write(pdb_path + "\n")

def process_pdb(pdb_file, start, end):
    """Process a pdb file:
       - For ATOM records, keep only atoms whose residue number is within [start, end].
       - For HETATM records, keep all heteroatoms except water (residue name "HOH").
       - Other lines (e.g. TER, END) remain unaltered.
    """
    try:
        with open(pdb_file, 'r') as f:
            lines = f.readlines()
    except Exception as e:
        print(f"Failed to open {pdb_file}: {e}")
        return False

    new_lines = []
    for line in lines:
        if line.startswith("ATOM"):
            # Residue number is typically in columns 23-26 (0-indexed 22:26)
            try:
                resnum_str = line[22:26].strip()
                resnum = int(resnum_str)
            except ValueError:
                # If conversion fails, keep the line unmodified.
                new_lines.append(line)
                continue

            # Keep atoms only within the specified residue range.
            if start <= resnum <= end:
                new_lines.append(line)
            else:
                # Skip atoms outside of the range.
                continue
        elif line.startswith("HETATM"):
            # Extract residue name (columns 18-20, index 17:20)
            resname = line[17:20].strip()
            if resname == "HOH":  # Skip water molecules
                continue
            else:
                new_lines.append(line)
        else:
            # Keep any other lines (e.g., TER, END) unchanged.
            new_lines.append(line)

    try:
        # Write the processed content back to the same file.
        with open(pdb_file, 'w') as f:
            f.writelines(new_lines)
        print(f"Processed: {pdb_file}")
        return True
    except Exception as e:
        print(f"Failed to write processed file {pdb_file}: {e}")
        return False

def main():
    processed_files = load_checkpoint(checkpoint_file)

    # Read the CSV file.
    with open(csv_file, newline='') as csvf:
        reader = csv.DictReader(csvf)
        for row in reader:
            try:
                start = int(row["start"])
                end = int(row["end"])
            except ValueError:
                print(f"Skipping row with invalid start/end values: {row}")
                continue

            source = row["source"].strip()
            structure = row["structure"].strip()

            # Only process rows from RCSB.
            if "RCSB" not in source:
                print(f"Skipping non-RCSB source: {source} for structure '{structure}'.")
                continue

            search_dir = rcsb_dir
            pdb_pattern = os.path.join(search_dir, f"*{structure}*.pdb")
            pdb_files = glob.glob(pdb_pattern)

            if not pdb_files:
                print(f"No PDB files found for pattern: {pdb_pattern}")
                continue

            for pdb_file in pdb_files:
                # Skip file if it has already been processed.
                if pdb_file in processed_files:
                    print(f"Skipping already processed file: {pdb_file}")
                    continue

                # Process the pdb file.
                if process_pdb(pdb_file, start, end):
                    update_checkpoint(checkpoint_file, pdb_file)
                    processed_files.add(pdb_file)

if __name__ == '__main__':
    main()

In [ ]:
import os
import glob

# Target directory for PDB files
TARGET_DIR = '/net/scratch/pkwon2/repeats_rcsb'

def count_residues(pdb_file):
    """
    Count unique protein residues in a PDB file.
    Each residue is identified by a tuple: (chain identifier, residue sequence number, insertion code)
    extracted from ATOM records.
    """
    residues = set()
    try:
        with open(pdb_file, 'r') as f:
            for line in f:
                if line.startswith("ATOM"):
                    # Chain identifier is usually at index 21,
                    # residue sequence number is in columns 23-26 (indices 22:26),
                    # and insertion code is at index 26.
                    chain = line[21].strip()
                    resseq = line[22:26].strip()
                    icode = line[26].strip()
                    residues.add((chain, resseq, icode))
    except Exception as e:
        print(f"Error reading {pdb_file}: {e}")
    return len(residues)

def main():
    pdb_files = glob.glob(os.path.join(TARGET_DIR, "*.pdb"))
    print(f"Found {len(pdb_files)} PDB files in '{TARGET_DIR}'.")
    
    for pdb_file in pdb_files:
        res_count = count_residues(pdb_file)
        if res_count > 9999:
            try:
                os.remove(pdb_file)
                print(f"Deleted '{pdb_file}' (residue count: {res_count}).")
            except Exception as e:
                print(f"Error deleting '{pdb_file}': {e}")
        else:
            print(f"Kept '{pdb_file}' (residue count: {res_count}).")

if __name__ == '__main__':
    main()

In [ ]:
#!/usr/bin/env python3
"""
1) Delete any PDB whose average B-factor ≥ b-threshold.
2) Remove HETATM entries not contacting protein within threshold Å.
3) Always delete any HETATM named 'UNK'.
4) Delete any PDB that raises an error.
Usage:
    python3 clean_rcsb.py \
      --rcsb-dir /net/scratch/pkwon2/repeats_rcsb \
      [--threshold 4.0] [--b-threshold 30.0]
"""

import os
import glob
import argparse
from Bio.PDB import PDBParser, NeighborSearch, PDBIO, Select

class HetFilter(Select):
    """Keep all ATOMs; for HETATMs only keep those contacting protein
    (within threshold Å) and not named 'UNK'."""
    def __init__(self, contacting):
        self._contacting = contacting

    def accept_atom(self, atom):
        is_het = atom.get_parent().get_id()[0] != " "
        if not is_het:
            return True

        # drop any 'UNK'
        if atom.get_parent().get_resname().strip().upper() == "UNK":
            return False

        # only keep if in the contacting set
        return atom in self._contacting

def process_pdb(pdb_path, threshold):
    """Run the HETATM filter and overwrite the file in place."""
    parser = PDBParser(QUIET=True)
    struct = parser.get_structure("", pdb_path)

    all_atoms     = list(struct.get_atoms())
    protein_atoms = [a for a in all_atoms if a.get_parent().get_id()[0] == " "]
    het_atoms     = [a for a in all_atoms if a.get_parent().get_id()[0] != " "]

    ns = NeighborSearch(protein_atoms)
    contacting = {het for het in het_atoms if ns.search(het.get_coord(), threshold, level="A")}

    io = PDBIO()
    io.set_structure(struct)
    tmp = pdb_path + ".tmp"
    io.save(tmp, HetFilter(contacting))
    os.replace(tmp, pdb_path)
    print(f"✔ Processed {os.path.basename(pdb_path)}")

def main():
    p = argparse.ArgumentParser(
        description="Clean RCSB PDBs: drop high‐B, remove non‐contacting HETATMs/UNK, delete errors."
    )
    p.add_argument("--rcsb-dir",
                   default="/net/scratch/pkwon2/repeats_rcsb",
                   help="Directory of RCSB PDB files")
    p.add_argument("--threshold", type=float, default=4.0,
                   help="Contact cutoff (Å) for HETATM‑protein")
    p.add_argument("--b-threshold", type=float, default=30.0,
                   help="Avg B‑factor cutoff (Å²); delete files with avg B-factor ≥ this")
    args, _ = p.parse_known_args()

    parser = PDBParser(QUIET=True)
    pattern = os.path.join(args.rcsb_dir, "*.pdb")

    for pdb_file in glob.glob(pattern):
        name = os.path.basename(pdb_file)

        # 1) Check average B-factor and delete if ≥ threshold
        try:
            struct = parser.get_structure(name, pdb_file)
            atoms = list(struct.get_atoms())
            if atoms:
                avg_b = sum(a.get_bfactor() for a in atoms) / len(atoms)
                if avg_b >= args.b_threshold:
                    os.remove(pdb_file)
                    print(f"→ Deleted {name}: avg B-factor {avg_b:.1f} ≥ {args.b_threshold}")
                    continue
        except Exception as e:
            print(f"✘ Parsing error in {name}: {e}")
            try:
                os.remove(pdb_file)
                print(f"→ Deleted {name} due to parse failure")
            except OSError:
                pass
            continue

        # 2) Filter HETATMs (and delete on error)
        try:
            process_pdb(pdb_file, args.threshold)
        except Exception as e:
            print(f"✘ Error in {name}: {e}")
            try:
                os.remove(pdb_file)
                print(f"→ Deleted {name} due to processing error")
            except OSError:
                print(f"⚠ Could not delete {name}")
            continue

if __name__ == "__main__":
    main()

In [ ]:
# #!/usr/bin/env python3
# """
# Clean and classify RCSB PDB files:

# 1) Delete PDBs whose average B-factor ≥ b-threshold.
# 2) Copy PDBs containing any calcium HETATM to /net/scratch/pkwon2/calcium_pdb.
# 3) Copy PDBs with non‑continuous protein residues to /net/scratch/pkwon2/noncontinous_pdb.
# 4) Remove HETATMs not contacting protein within threshold Å.
# 5) Always delete HETATMs named 'UNK'.
# 6) Delete any PDB that raises a parsing or processing error.
# Usage:
#     python3 clean_and_classify_rcsb.py \
#       --rcsb-dir /net/scratch/pkwon2/repeats_rcsb \
#       [--threshold 4.0] [--b-threshold 30.0]
# """
# import os
# import glob
# import argparse
# import shutil
# from Bio.PDB import PDBParser, NeighborSearch, PDBIO, Select

# class HetFilter(Select):
#     """Keep all ATOMs; for HETATMs only keep those contacting protein
#     (within threshold Å) and not named 'UNK'."""
#     def __init__(self, contacting):
#         self._contacting = contacting

#     def accept_atom(self, atom):
#         is_het = atom.get_parent().get_id()[0] != " "
#         if not is_het:
#             return True
#         # drop any 'UNK'
#         if atom.get_parent().get_resname().strip().upper() == "UNK":
#             return False
#         # only keep if in the contacting set
#         return atom in self._contacting

# def process_pdb(pdb_path, threshold):
#     """Filter HETATMs and overwrite the file in place."""
#     parser = PDBParser(QUIET=True)
#     struct = parser.get_structure("", pdb_path)

#     all_atoms     = list(struct.get_atoms())
#     protein_atoms = [a for a in all_atoms if a.get_parent().get_id()[0] == " "]
#     het_atoms     = [a for a in all_atoms if a.get_parent().get_id()[0] != " "]

#     ns = NeighborSearch(protein_atoms)
#     contacting = {het for het in het_atoms if ns.search(het.get_coord(), threshold, level="A")}

#     io = PDBIO()
#     io.set_structure(struct)
#     tmp = pdb_path + ".tmp"
#     io.save(tmp, HetFilter(contacting))
#     os.replace(tmp, pdb_path)
#     print(f"✔ Processed {os.path.basename(pdb_path)}")

# def contains_calcium(struct):
#     """Return True if any HETATM residue is calcium ('CA')."""
#     for res in struct.get_residues():
#         if res.id[0] != " " and res.get_resname().strip().upper() == "CA":
#             return True
#     return False

# def is_noncontinuous(struct):
#     """
#     Return True if any chain has missing residue numbers
#     between its min and max resseq (insertion codes ignored).
#     """
#     for chain in struct.get_chains():
#         prot_res = [r for r in chain.get_residues() if r.id[0] == " "]
#         if not prot_res:
#             continue
#         nums = [r.id[1] for r in prot_res]
#         if len(set(nums)) != max(nums) - min(nums) + 1:
#             return True
#     return False

# def main():
#     p = argparse.ArgumentParser(
#         description="Clean and classify RCSB PDBs."
#     )
#     p.add_argument("--rcsb-dir",
#                    default="/net/scratch/pkwon2/repeats_rcsb",
#                    help="Directory of RCSB PDB files")
#     p.add_argument("--threshold", type=float, default=4.0,
#                    help="Contact cutoff (Å) for HETATM–protein")
#     p.add_argument("--b-threshold", type=float, default=30.0,
#                    help="Avg B‑factor cutoff (Å²); delete files with avg B ≥ this")
#     args, _ = p.parse_known_args()

#     calcium_dir   = "/net/scratch/pkwon2/calcium_pdb"
#     noncont_dir   = "/net/scratch/pkwon2/noncontinous_pdb"
#     os.makedirs(calcium_dir, exist_ok=True)
#     os.makedirs(noncont_dir, exist_ok=True)

#     parser_pdb = PDBParser(QUIET=True)
#     pattern    = os.path.join(args.rcsb_dir, "*.pdb")

#     for pdb_file in glob.glob(pattern):
#         name = os.path.basename(pdb_file)

#         # 1) Initial parse
#         try:
#             struct = parser_pdb.get_structure(name, pdb_file)
#         except Exception as e:
#             print(f"✘ Parsing error in {name}: {e}")
#             try:
#                 os.remove(pdb_file)
#                 print(f"→ Deleted {name} due to parse failure")
#             except OSError:
#                 pass
#             continue

#         # 2) B‑factor check
#         atoms = list(struct.get_atoms())
#         if atoms:
#             avg_b = sum(a.get_bfactor() for a in atoms) / len(atoms)
#             if avg_b >= args.b_threshold:
#                 os.remove(pdb_file)
#                 print(f"→ Deleted {name}: avg B-factor {avg_b:.1f} ≥ {args.b_threshold}")
#                 continue

#         # 3) Classification: calcium-containing
#         if contains_calcium(struct):
#             try:
#                 shutil.copy2(pdb_file, calcium_dir)
#                 print(f"→ Copied {name} to calcium_pdb")
#             except Exception as err:
#                 print(f"⚠ Could not copy {name} to calcium_pdb: {err}")

#         # 4) Classification: non-continuous protein
#         if is_noncontinuous(struct):
#             try:
#                 shutil.copy2(pdb_file, noncont_dir)
#                 print(f"→ Copied {name} to noncontinous_pdb")
#             except Exception as err:
#                 print(f"⚠ Could not copy {name} to noncontinous_pdb: {err}")

#         # 5) Clean HETATMs
#         try:
#             process_pdb(pdb_file, args.threshold)
#         except Exception as e:
#             print(f"✘ Error processing {name}: {e}")
#             try:
#                 os.remove(pdb_file)
#                 print(f"→ Deleted {name} due to processing error")
#             except OSError:
#                 print(f"⚠ Could not delete {name}")
#             continue

# if __name__ == "__main__":
#     main()

In [ ]:
# #!/usr/bin/env python3
# """
# Delete non‑continuous PDBs and remove hydrogens from the rest.

# 1) For each PDB in --rcsb-dir:
#    • If any chain’s protein residues (ATOM records) have gaps
#      in their sequence numbering, delete the file.
#    • Otherwise, strip out all hydrogen atoms and overwrite the file.

# Usage:
#     python3 clean_repeats_rcsb.py \
#         --rcsb-dir /net/scratch/pkwon2/repeats_rcsb
# """
# import os
# import glob
# import argparse
# from Bio.PDB import PDBParser, PDBIO, Select

# class NoHydrogen(Select):
#     """Select only non‑hydrogen atoms."""
#     def accept_atom(self, atom):
#         # Prefer the element attribute if present
#         el = atom.element.strip() if atom.element else ''
#         if el.upper() == 'H':
#             return False
#         # Fallback: atom name starts with 'H'
#         if atom.get_name().strip().startswith('H'):
#             return False
#         return True

# def is_noncontinuous(structure):
#     """
#     Return True if any chain has missing residue numbers
#     between its minimum and maximum.
#     """
#     for chain in structure.get_chains():
#         # collect only protein residues (hetero flag == ' ')
#         nums = [res.id[1]
#                 for res in chain.get_residues()
#                 if res.id[0] == ' ']
#         if not nums:
#             continue
#         if set(nums) != set(range(min(nums), max(nums) + 1)):
#             return True
#     return False

# def main():
#     p = argparse.ArgumentParser(description="Clean repeats_rcsb PDBs")
#     p.add_argument(
#         "--rcsb-dir",
#         default="/net/scratch/pkwon2/repeats_rcsb",
#         help="Directory containing your RCSB PDB files"
#     )
#     args, _ = p.parse_known_args()

#     parser = PDBParser(QUIET=True)
#     pattern = os.path.join(args.rcsb_dir, "*.pdb")

#     for pdb_path in glob.glob(pattern):
#         name = os.path.basename(pdb_path)
#         # 1) Parse structure
#         try:
#             struct = parser.get_structure(name, pdb_path)
#         except Exception as e:
#             print(f"✘ Failed to parse {name}: {e}")
#             continue

#         # 2) Delete if non‑continuous
#         if is_noncontinuous(struct):
#             try:
#                 os.remove(pdb_path)
#                 print(f"→ Deleted non‑continuous {name}")
#             except OSError as e:
#                 print(f"⚠ Could not delete {name}: {e}")
#             continue

#         # 3) Remove hydrogens from continuous PDB
#         try:
#             io = PDBIO()
#             io.set_structure(struct)
#             tmp = pdb_path + ".tmp"
#             io.save(tmp, NoHydrogen())
#             os.replace(tmp, pdb_path)
#             print(f"✔ Stripped hydrogens from {name}")
#         except Exception as e:
#             print(f"✘ Error stripping hydrogens from {name}: {e}")

# if __name__ == "__main__":
#     main()

#!/usr/bin/env python3
"""
Keep non-continuous PDBs and remove hydrogens from the rest.

1) For each PDB in --rcsb-dir:
   • If any chain’s protein residues (ATOM records) have gaps
     in their sequence numbering, leave the file as-is.
   • Otherwise, strip out all hydrogen atoms and overwrite the file.

Usage:
    python3 clean_repeats_rcsb.py \
        --rcsb-dir /net/scratch/pkwon2/repeats_rcsb
"""
import os
import glob
import argparse
from Bio.PDB import PDBParser, PDBIO, Select

class NoHydrogen(Select):
    """Select only non-hydrogen atoms."""
    def accept_atom(self, atom):
        # Prefer the element attribute if present
        el = atom.element.strip() if atom.element else ''
        if el.upper() == 'H':
            return False
        # Fallback: atom name starts with 'H'
        if atom.get_name().strip().startswith('H'):
            return False
        return True

def is_noncontinuous(structure):
    """
    Return True if any chain has missing residue numbers
    between its minimum and maximum.
    """
    for chain in structure.get_chains():
        # collect only protein residues (hetero flag == ' ')
        nums = [res.id[1]
                for res in chain.get_residues()
                if res.id[0] == ' ']
        if not nums:
            continue
        if set(nums) != set(range(min(nums), max(nums) + 1)):
            return True
    return False

def main():
    p = argparse.ArgumentParser(description="Clean repeats_rcsb PDBs")
    p.add_argument(
        "--rcsb-dir",
        default="/net/scratch/pkwon2/repeats_rcsb",
        help="Directory containing your RCSB PDB files"
    )
    args, _ = p.parse_known_args()

    parser = PDBParser(QUIET=True)
    pattern = os.path.join(args.rcsb_dir, "*.pdb")

    for pdb_path in glob.glob(pattern):
        name = os.path.basename(pdb_path)
        # 1) Parse structure
        try:
            struct = parser.get_structure(name, pdb_path)
        except Exception as e:
            print(f"✘ Failed to parse {name}: {e}")
            continue

        # 2) If non-continuous, leave it alone
        if is_noncontinuous(struct):
            print(f"→ Kept non-continuous {name}")
            continue

        # 3) Otherwise strip hydrogens in place
        try:
            io = PDBIO()
            io.set_structure(struct)
            tmp = pdb_path + ".tmp"
            io.save(tmp, NoHydrogen())
            os.replace(tmp, pdb_path)
            print(f"✔ Stripped hydrogens from continuous {name}")
        except Exception as e:
            print(f"✘ Error stripping hydrogens from {name}: {e}")

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
"""
Clean RCSB PDB files:

1) Delete PDBs whose average B-factor ≥ b-threshold.
2) Always delete HETATMs named 'UNK'.
3) Delete any PDB that raises a parsing or processing error.

Usage:
    python3 clean_rcsb.py \
      --rcsb-dir /net/scratch/pkwon2/repeats_rcsb \
      [--b-threshold 30.0]
"""
import os
import glob
import argparse
from Bio.PDB import PDBParser, PDBIO, Select

class HetFilter(Select):
    """Keep all ATOMs; for HETATMs only drop those named 'UNK'."""
    def accept_atom(self, atom):
        parent = atom.get_parent()
        is_het = parent.get_id()[0] != " "
        if not is_het:
            return True
        # drop any 'UNK'
        return parent.get_resname().strip().upper() != "UNK"

def process_pdb(pdb_path):
    """Strip UNK HETATMs and overwrite the file in place."""
    parser = PDBParser(QUIET=True)
    struct = parser.get_structure("", pdb_path)

    io = PDBIO()
    io.set_structure(struct)
    tmp = pdb_path + ".tmp"
    io.save(tmp, HetFilter())
    os.replace(tmp, pdb_path)
    print(f"✔ Cleaned {os.path.basename(pdb_path)}")

def main():
    p = argparse.ArgumentParser(description="Clean RCSB PDBs.")
    p.add_argument("--rcsb-dir",
                   default="/net/scratch/pkwon2/repeats_rcsb",
                   help="Directory of RCSB PDB files")
    p.add_argument("--b-threshold", type=float, default=30.0,
                   help="Avg B‑factor cutoff (Å²); delete files with avg B ≥ this")

    # ignore any unknown args (e.g. Jupyter’s -f)
    args, _ = p.parse_known_args()

    parser = PDBParser(QUIET=True)
    pattern = os.path.join(args.rcsb_dir, "*.pdb")

    for pdb_file in glob.glob(pattern):
        name = os.path.basename(pdb_file)

        # 1) Initial parse
        try:
            struct = parser.get_structure(name, pdb_file)
        except Exception as e:
            print(f"✘ Parsing error in {name}: {e}")
            try:
                os.remove(pdb_file)
                print(f"→ Deleted {name} due to parse failure")
            except OSError:
                pass
            continue

        # 2) B‑factor check
        atoms = list(struct.get_atoms())
        if atoms:
            avg_b = sum(a.get_bfactor() for a in atoms) / len(atoms)
            if avg_b >= args.b_threshold:
                os.remove(pdb_file)
                print(f"→ Deleted {name}: avg B-factor {avg_b:.1f} ≥ {args.b_threshold}")
                continue

        # 3) Strip UNK HETATMs
        try:
            process_pdb(pdb_file)
        except Exception as e:
            print(f"✘ Error processing {name}: {e}")
            try:
                os.remove(pdb_file)
                print(f"→ Deleted {name} due to processing error")
            except OSError:
                pass

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
"""
Strip non‑contacting HETATM residues from PDBs in a directory.

For each PDB:
  • Parse structure.
  • Identify all protein atoms (ATOM records).
  • For each HETATM residue, test whether any atom in that residue
    is within the cutoff of any protein atom.
  • Keep all ATOM records, and only those HETATM residues that contact
    the protein.  Overwrite the PDB in place.

Usage:
    python3 strip_noncontacting_het.py \
      --rcsb-dir /net/scratch/pkwon2/repeats_rcsb \
      [--threshold 4.0]
"""
import os
import glob
import argparse

from Bio.PDB import PDBParser, NeighborSearch, PDBIO, Select

class KeepContactingHet(Select):
    """Select only:
       - All ATOM residues (protein).
       - HETATM residues whose resid tuple appears in contacting_residues.
    """
    def __init__(self, contacting_residues):
        self.contacting = set(contacting_residues)

    def accept_residue(self, residue):
        hetflag = residue.id[0] != " "
        if not hetflag:
            # always keep protein residues
            return True
        # keep only HET residues that contacted
        return (residue.get_full_id() in self.contacting)

def clean_pdb(pdb_path, cutoff):
    """Parse, compute contacting het residues, and overwrite file."""
    parser = PDBParser(QUIET=True)
    struct = parser.get_structure("", pdb_path)

    # collect protein atoms
    prot_atoms = [a for a in struct.get_atoms()
                  if a.get_parent().get_id()[0] == " "]
    ns = NeighborSearch(prot_atoms)

    # detect which het residues contact the protein
    contacting = []
    for model in struct:
        for chain in model:
            for res in chain:
                if res.id[0] != " ":
                    # any atom in this het residue contacting?
                    for atom in res:
                        if ns.search(atom.coord, cutoff, level="A"):
                            contacting.append(res.get_full_id())
                            break

    # write out only protein + contacting het residues
    io = PDBIO()
    io.set_structure(struct)
    tmp = pdb_path + ".tmp"
    io.save(tmp, KeepContactingHet(contacting))
    os.replace(tmp, pdb_path)
    print(f"✔ {os.path.basename(pdb_path)} cleaned (kept {len(contacting)} het residues)")

def main():
    p = argparse.ArgumentParser(
        description="Strip non‑contacting HETATMs from PDB files."
    )
    p.add_argument(
        "--rcsb-dir",
        default="/net/scratch/pkwon2/repeats_rcsb",
        help="Directory containing .pdb files"
    )
    p.add_argument(
        "--threshold", type=float, default=4.0,
        help="Distance cutoff (Å) to define HET–protein contact"
    )
    args, _ = p.parse_known_args()

    pattern = os.path.join(args.rcsb_dir, "*.pdb")
    for pdb_file in glob.glob(pattern):
        try:
            clean_pdb(pdb_file, args.threshold)
        except Exception as e:
            print(f"✘ Error in {os.path.basename(pdb_file)}: {e}")

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
import os
import glob

# directory containing your PDBs
PDB_DIR = "/net/scratch/pkwon2/repeats_rcsb"

def reindex_and_relabel(pdb_path):
    """
    Rewrites the PDB file in place:
      - renumbers ATOM records from 1 upward,
      - then renumbers HETATM records continuing the count,
      - sets chain ID to 'A' on all ATOM/HETATM lines,
      - leaves all other lines untouched.
    """
    with open(pdb_path, 'r') as f:
        lines = f.readlines()

    new_lines = []
    serial = 1

    for line in lines:
        if line.startswith("ATOM  ") or line.startswith("HETATM"):
            # rebuild the record:
            # - columns 1-6  (0:6): record name
            # - columns 7-11 (6:11): new serial, right‑justified
            # - columns 12-21 (11:21): keep as is (atom name, residue, etc.)
            # - column 22    (21): chain ID -> 'A'
            # - columns 23+  (22:): keep as is
            rec_name = line[:6]
            rest_pre_chain = line[11:21]
            rest_post_chain = line[22:]
            new_serial = f"{serial:>5}"
            new_line = rec_name + new_serial + rest_pre_chain + "A" + rest_post_chain
            new_lines.append(new_line)
            serial += 1
        else:
            # leave all other lines (TER, CONECT, HEADER, etc.) unchanged
            new_lines.append(line)

    # overwrite the file
    with open(pdb_path, 'w') as f:
        f.writelines(new_lines)
    print(f"Processed {os.path.basename(pdb_path)}: total ATOM+HETATM = {serial-1}")

def main():
    pattern = os.path.join(PDB_DIR, "*.pdb")
    pdb_files = glob.glob(pattern)
    if not pdb_files:
        print(f"No PDB files found in {PDB_DIR}")
        return

    for pdb in pdb_files:
        reindex_and_relabel(pdb)

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
"""
Script to renumber residues sequentially and relabel chain IDs to 'A' for PDB files.
- Residue sequence numbers (columns 23-26) are reassigned starting at 1 for the first unique residue encountered,
  incrementing by 1 whenever the original residue identity changes.
- Chain ID (column 22) is forced to 'A'.
- All other fields and lines (including atom serial numbers) remain unchanged.
Processes all .pdb files in /net/scratch/pkwon2/repeats_rcsb in place.
"""
import os
import glob

# Directory containing your PDB files
directory = '/net/scratch/pkwon2/repeats_rcsb'

# Iterate through each PDB file in the directory
def renumber_residues(pdb_path):
    with open(pdb_path, 'r') as f:
        lines = f.readlines()

    residue_map = {}  # maps (resName, origResSeq) -> new sequential index
    next_index = 1
    new_lines = []

    for line in lines:
        if line.startswith(('ATOM  ', 'HETATM')):
            # Extract original residue identity
            res_name = line[17:20]
            orig_resseq = line[22:26].strip()
            key = (res_name, orig_resseq)
            # Assign new index if unseen
            if key not in residue_map:
                residue_map[key] = next_index
                next_index += 1
            # Build new residue sequence field (4 chars, right-justified)
            new_resseq = f"{residue_map[key]:>4}"
            # Force chain ID to 'A'
            # Columns: 0-21 includes up to col22 (index21) exclusive
            prefix = line[:21]
            # Construct new line: prefix + 'A' (chain) + new_resseq + remainder from col27 onward
            new_line = prefix + 'A' + new_resseq + line[26:]
            new_lines.append(new_line)
        else:
            # leave non-ATOM/HETATM lines unchanged
            new_lines.append(line)

    # Overwrite the original file
    with open(pdb_path, 'w') as f:
        f.writelines(new_lines)

    print(f"Renumbered {os.path.basename(pdb_path)}: {next_index-1} residues assigned.")

if __name__ == '__main__':
    pattern = os.path.join(directory, '*.pdb')
    for pdb_file in glob.glob(pattern):
        renumber_residues(pdb_file)


In [ ]:
#!/usr/bin/env python3
import os
import glob

# Directory containing AF2 PDB files
PDB_DIR = '/net/scratch/pkwon2/repeat_proteins_clean/repeats_af2'
# Files with average pLDDT <= this will be deleted
THRESHOLD = 90.0

def average_plddt(pdb_path):
    """Read ATOM lines in a PDB and return the average B‑factor (pLDDT)."""
    scores = []
    with open(pdb_path, 'r') as f:
        for line in f:
            if line.startswith('ATOM'):
                # PDB format: B‑factor is columns 61–66 (1‑based), i.e. line[60:66] in Python
                try:
                    plddt = float(line[60:66].strip())
                    scores.append(plddt)
                except ValueError:
                    # skip lines where B‑factor isn’t a float
                    continue
    if not scores:
        return None
    return sum(scores) / len(scores)

def main():
    pattern = os.path.join(PDB_DIR, '*.pdb')
    for pdb_file in glob.glob(pattern):
        avg = average_plddt(pdb_file)
        if avg is None:
            print(f"[SKIP] No ATOM records found in {os.path.basename(pdb_file)}")
            continue
        if avg <= THRESHOLD:
            os.remove(pdb_file)
            print(f"[DELETE] {os.path.basename(pdb_file)} (avg pLDDT = {avg:.2f})")
        else:
            print(f"[KEEP]   {os.path.basename(pdb_file)} (avg pLDDT = {avg:.2f})")

if __name__ == '__main__':
    main()

In [ ]:
#!/usr/bin/env python3
import glob
import os

# Directory containing your PDB files
rcsb_dir = '/net/scratch/pkwon2/repeats_rcsb'

# Find all .pdb files in that directory
pdb_paths = glob.glob(os.path.join(rcsb_dir, '*.pdb'))

for pdb_path in pdb_paths:
    # Read the original file
    with open(pdb_path, 'r') as infile:
        lines = infile.readlines()

    # Filter out any lines whose record name is TER or END
    filtered = [
        line for line in lines
        if not (line.startswith('TER') or line.startswith('END'))
    ]

    # Overwrite the original file with the filtered lines
    with open(pdb_path, 'w') as outfile:
        outfile.writelines(filtered)

    print(f"Cleaned {os.path.basename(pdb_path)}")

In [ ]:
#!/usr/bin/env python3
"""
Separate PDBs with and without HETATM records.

* Source directory is hard-coded to:
    /net/scratch/pkwon2/repeat_proteins_clean/repeats_rcsb

* Two sub-directories are created automatically (if absent):
    ├── with_hetatm/      – files containing at least one HETATM line
    └── without_hetatm/   – files containing no HETATM lines

The script copies (does not move) the files and preserves timestamps.
"""

import os
import glob
import shutil
from pathlib import Path

# ------------ paths ------------
SRC_DIR = Path("/projects/ml/pkwon2")
WITH_HET_DIR = SRC_DIR / "with_hetatm"
WITHOUT_HET_DIR = SRC_DIR / "without_hetatm"

# Ensure target folders exist
WITH_HET_DIR.mkdir(exist_ok=True)
WITHOUT_HET_DIR.mkdir(exist_ok=True)

# ------------ processing ------------
for pdb_path_str in glob.glob(str(SRC_DIR / "*.pdb")):
    pdb_path = Path(pdb_path_str)
    has_hetatm = False

    # Peek line-by-line until we know whether a HETATM record exists
    with pdb_path.open() as fh:
        for line in fh:
            if line.startswith("HETATM"):
                has_hetatm = True
                break

    dest_dir = WITH_HET_DIR if has_hetatm else WITHOUT_HET_DIR
    shutil.copy2(pdb_path, dest_dir)

    print(f"{'WITH' if has_hetatm else 'WITHOUT'} HETATM → {pdb_path.name}")

print("Done.")

In [ ]:
#!/usr/bin/env python3
import os
import glob

# --- adjust these paths if needed ---
PDB_DIR   = '/projects/ml/pkwon2/repeat_protein_data'
OUT_FASTA = '/projects/ml/pkwon2/repeat_protein_data.fasta'
# -------------------------------------

# three‑letter → one‑letter map
AA_MAP = {
    'ALA':'A','CYS':'C','ASP':'D','GLU':'E','PHE':'F',
    'GLY':'G','HIS':'H','ILE':'I','LYS':'K','LEU':'L',
    'MET':'M','MSE':'M','ASN':'N','PRO':'P','GLN':'Q',
    'ARG':'R','SER':'S','THR':'T','VAL':'V','TRP':'W',
    'TYR':'Y'
}

def parse_seqres(pdb_path):
    chains = {}
    with open(pdb_path) as fp:
        for line in fp:
            if line.startswith('SEQRES'):
                chain = line[11]
                res   = line[19:].split()
                chains.setdefault(chain, []).extend(res)
    return chains

def parse_atom_seq(pdb_path):
    seen = set()
    chains = {}
    with open(pdb_path) as fp:
        for line in fp:
            if line.startswith('ATOM  '):
                resname = line[17:20].strip()
                chain   = line[21]
                resnum  = int(line[22:26])
                icode   = line[26]
                key     = (chain, resnum, icode)
                if key not in seen:
                    seen.add(key)
                    chains.setdefault(chain, []).append(resname)
    return chains

def write_fasta_entry(fh, pdb_id, chain_id, residues):
    seq = ''.join(AA_MAP.get(r,'X') for r in residues)
    fh.write(f'>{pdb_id}_{chain_id}\n')
    for i in range(0, len(seq), 60):
        fh.write(seq[i:i+60] + '\n')

def main():
    # remove old output if it exists
    if os.path.exists(OUT_FASTA):
        os.remove(OUT_FASTA)

    pdb_paths = glob.glob(os.path.join(PDB_DIR, '*.pdb'))
    if not pdb_paths:
        print(f'No PDB files found in {PDB_DIR}')
        return

    written = 0
    deleted = 0

    with open(OUT_FASTA, 'w') as out_fh:
        for pdb_path in pdb_paths:
            pdb_id = os.path.splitext(os.path.basename(pdb_path))[0]

            # 1) try SEQRES
            chains = parse_seqres(pdb_path)
            source = 'SEQRES'

            # 2) fallback to ATOM if no SEQRES
            if not chains:
                chains = parse_atom_seq(pdb_path)
                source = 'ATOM' if chains else None

            # 3) if still nothing, delete file and skip
            if not source:
                print(f'Deleting {pdb_id}.pdb (no SEQRES or ATOM records)')
                os.remove(pdb_path)
                deleted += 1
                continue

            # 4) write out each chain
            for chain_id, residues in chains.items():
                write_fasta_entry(out_fh, pdb_id, chain_id, residues)
                written += 1

    print(f'Done: wrote {written} chain FASTA entries; deleted {deleted} empty PDB files.')

if __name__ == '__main__':
    main()

In [ ]:
#!/bin/bash
#SBATCH -p cpu
#SBATCH -J mmseqs_iiab
#SBATCH --time=02:00:00
#SBATCH -c 8
#SBATCH --mem=12g
#SBATCH -o digs.log
#SBATCH -e digs.log

/software/mmseqs2/bin/mmseqs easy-linclust PATH/TO/INPUT/FILE.fasta PATH/TO/OUTPUT/PREFIX /tmp --min-seq-id 0.98 -c 0.8 --cov-mode 0 --threads 8

In [ ]:
#!/bin/bash
#SBATCH -p cpu
#SBATCH -J mmseqs_iiab
#SBATCH --time=02:00:00
#SBATCH -c 8
#SBATCH --mem=12g
#SBATCH -o digs.log
#SBATCH -e digs.log

/software/mmseqs2/bin/mmseqs easy-linclust /projects/ml/pkwon2/repeat_protein_data.fasta /projects/ml/pkwon2/ /tmp --min-seq-id 0.98 -c 0.8 --cov-mode 0 --threads 8

In [ ]:
#!/usr/bin/env python3
import os
import shutil

# --- CONFIGURATION ---
cluster_tsv   = '/projects/ml/pkwon2/_cluster.tsv'
merged_dir    = '/projects/ml/pkwon2/repeat_protein_data'
output_root   = '/projects/ml/pkwon2/training_repeat_protein_data'

# make top‐level output dir
os.makedirs(output_root, exist_ok=True)

with open(cluster_tsv, 'r') as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith('#'):
            continue

        # split into cluster name and member ID
        cluster_id, member_id = line.split()

        # derive the actual filename: strip one "_A" and add .pdb
        if member_id.endswith('_A'):
            stem = member_id[:-2]
        else:
            stem = member_id
        pdb_fn = stem + '.pdb'

        # source & destination paths
        src = os.path.join(merged_dir, pdb_fn)
        dst_dir = os.path.join(output_root, cluster_id)
        dst = os.path.join(dst_dir, pdb_fn)

        # ensure cluster directory exists
        os.makedirs(dst_dir, exist_ok=True)

        # copy if present, warn otherwise
        if os.path.exists(src):
            shutil.copy(src, dst)
        else:
            print(f'Warning: {src} not found, skipping.')

print("Done.")

In [2]:
#!/usr/bin/env python3
import os
import pandas as pd

root_dir = '/projects/ml/pkwon2/training_repeat_protein_data'
records = []

for cluster_name in os.listdir(root_dir):
    cluster_dir = os.path.join(root_dir, cluster_name)
    if not os.path.isdir(cluster_dir):
        continue

    pdb_files = [f for f in os.listdir(cluster_dir) if f.endswith('.pdb')]
    if not pdb_files:
        continue

    af_files     = [f for f in pdb_files if f.startswith('AF-')]
    non_af_files = [f for f in pdb_files if not f.startswith('AF-')]

    if af_files and non_af_files:
        include = non_af_files
    else:
        include = pdb_files

    for fn in include:
        records.append({
            'path':       os.path.join(cluster_dir, fn),
            'example_id': os.path.splitext(fn)[0],
            'cluster':    cluster_name
        })

df = pd.DataFrame(records, columns=['path','example_id','cluster'])

# <-- specify engine now that pyarrow is installed -->
df.to_parquet(
    '/projects/ml/pkwon2/training_repeat_protein_data/repeat_protein.parquet',
    engine='pyarrow',
    index=False
)

print(f"Wrote {len(df)} rows.")

Wrote 28262 rows.


In [ ]:
pip install --user pyarrow

In [4]:
import pandas as pd

# point to your parquet
fn = '/projects/ml/pkwon2/training_repeat_protein_data/repeat_protein.parquet'
# this will pull it into a DataFrame
df = pd.read_parquet(fn, engine='pyarrow')

# show the first few rows
print(df.head().to_string())

# or, to dump the entire thing neatly:
print(df.to_string())

df.to_csv('/projects/ml/pkwon2/repeat_protein.csv', index=False)

                                                                                                     path               example_id                    cluster
0                                    /projects/ml/pkwon2/training_repeat_protein_data/6lu8_6_A/3j2i_B.pdb                   3j2i_B                   6lu8_6_A
1                                    /projects/ml/pkwon2/training_repeat_protein_data/6lu8_6_A/6lu8_6.pdb                   6lu8_6                   6lu8_6_A
2                                    /projects/ml/pkwon2/training_repeat_protein_data/6lu8_6_A/2x7n_B.pdb                   2x7n_B                   6lu8_6_A
3                                    /projects/ml/pkwon2/training_repeat_protein_data/6lu8_6_A/5m3q_A.pdb                   5m3q_A                   6lu8_6_A
4  /projects/ml/pkwon2/training_repeat_protein_data/AF-Q7TMQ7-F1-model_v4_A_A/AF-Q5ZLL7-F1-model_v4_A.pdb  AF-Q5ZLL7-F1-model_v4_A  AF-Q7TMQ7-F1-model_v4_A_A


IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [ ]:
#!/usr/bin/env python3
"""
Clean out HETATM groups larger than `MAX_GROUP_SIZE` atoms
from all non-AF* PDBs under:
/net/scratch/pkwon2/training_repeat_protein_data/*/*.pdb

Keeps:
  - All ATOM records
  - Any HETATM group whose atom count ≤ MAX_GROUP_SIZE
  - All other lines (HEADER, TER, etc.)

Prints which HETATM groups were deleted per file.
"""

import glob
import os
import shutil
from collections import defaultdict

# ——— CONFIG —————————————————————————————————————
PATTERN          = "/projects/ml/pkwon2/training_repeat_protein_data/*/*.pdb"
SKIP_PREFIX      = "AF"
MAX_GROUP_SIZE   = 5   # keep HETATM groups of this size or smaller
# —————————————————————————————————————————————————

def process_pdb(path):
    bak = path + ".bak"
    if not os.path.exists(bak):
        shutil.copy2(path, bak)

    lines = open(path, "r").readlines()

    # Group HETATM lines by (resName, chainID, resSeq, iCode)
    het_groups = defaultdict(list)
    for i, L in enumerate(lines):
        if L.startswith("HETATM"):
            key = (L[17:20].strip(), L[21], L[22:26].strip(), L[26])
            het_groups[key].append(i)

    # Identify which groups exceed threshold
    to_delete = { key: idxs
                  for key, idxs in het_groups.items()
                  if len(idxs) > MAX_GROUP_SIZE }

    # Rewrite, dropping all lines in to_delete groups
    out = []
    for i, L in enumerate(lines):
        if L.startswith("HETATM"):
            for key, idxs in to_delete.items():
                if i in idxs:
                    break
            else:
                out.append(L)
        else:
            out.append(L)

    with open(path, "w") as f:
        f.writelines(out)

    # Print report
    if to_delete:
        print(f"{os.path.basename(path)}: removed {len(to_delete)} HETATM group(s):")
        for (resname, chain, resseq, icode), idxs in to_delete.items():
            count = len(idxs)
            loc = f"{chain}{resseq}{icode}".strip()
            print(f"  • {resname} {loc} → {count} atoms")
    else:
        print(f"{os.path.basename(path)}: no HETATM groups > {MAX_GROUP_SIZE} atoms")

def main():
    for pdb in glob.iglob(PATTERN, recursive=True):
        if os.path.basename(pdb).startswith(SKIP_PREFIX):
            continue
        process_pdb(pdb)

if __name__ == "__main__":
    main()